<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Gradient boosting with LightGBM

**[Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/)** · Forecasting and Time Series: Predict What Comes Next · Module 5, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train one LightGBM model for all products with mlforecast, read its feature importance, forecast 14 days ahead in three ways (recursive, lags of 14 days and more, direct), and find the leak in a rolling feature.

| | |
|---|---|
| **Time** | About 45 minutes, of which about 2 minutes of waiting for the backtests |
| **Needs** | An internet connection for the setup cells. No account and no key. The first cell also installs mlforecast 1.1.0 and LightGBM 4.7.0. |
| **You should know** | Gradient boosting, from the [machine learning course](https://learning.nextia-ai.com/courses/ml/). Lags, rolling means and a global model, from [Forecasting as regression](https://learning.nextia-ai.com/courses/forecasting/m05/forecasting-as-regression/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M05-L02-gradient-boosting-with-lightgbm/gradient-boosting-with-lightgbm-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, lightgbm 4.7.0, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 mlforecast==1.1.0 lightgbm==4.7.0 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import mlforecast; print("mlforecast", mlforecast.__version__)
import lightgbm; print("lightgbm", lightgbm.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One LightGBM model for all products

The course's machine learning rows use the same inputs: lags of 1, 7, 14 and 28 days, rolling means, the weekday, the month, the day of the year, the price, the promotion flag and three holiday flags from `calendar.csv`. The next cell builds the holiday flags, trains LightGBM at the last cut-off and prints the share of each feature in the model's **gain**: how much the splits on that feature reduced the error.

> **Predict.** Which feature will LightGBM use most: the promotion flag, the weekday or a recent mean? Then run the cell.

In [ ]:
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean

cal = pd.read_csv("data/calendar.csv", parse_dates=["date"])
cal["black_friday_week"] = 0
for day in cal.loc[cal["holiday"] == "Black Friday", "date"]:
    cal.loc[(cal["date"] >= day) & (cal["date"] <= day + pd.Timedelta(days=3)), "black_friday_week"] = 1
cal["christmas_run_up"] = ((cal["date"].dt.month == 12) & (cal["date"].dt.day <= 23)).astype(int)
EXOG = ["price", "on_promotion", "is_holiday", "black_friday_week", "christmas_run_up"]
sales = load_sales(BACKTEST_PRODUCTS).merge(
    cal[["date", "is_holiday", "black_friday_week", "christmas_run_up"]].rename(columns={"date": "ds"}), on="ds", how="left")

LGB = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=20, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.8, random_state=35, n_jobs=2, verbose=-1)

def make_model(lags, lag_transforms):
    return MLForecast(models={"lgbm": lgb.LGBMRegressor(**LGB)}, freq="D", lags=lags, lag_transforms=lag_transforms,
                      date_features=["dayofweek", "month", "dayofyear"], num_threads=1)

RECURSIVE = dict(lags=[1, 7, 14, 28], lag_transforms={1: [RollingMean(7), RollingMean(28)], 7: [RollingMean(4)]})
m = make_model(**RECURSIVE)
m.fit(sales[sales["ds"] <= ORIGINS[-1]][["unique_id", "ds", "y"] + EXOG], static_features=[])
gain = pd.Series(m.models_["lgbm"].booster_.feature_importance("gain"), index=m.ts.features_order_)
share = (100 * gain / gain.sum()).sort_values(ascending=False).round(1)
print(share.head(10).to_string())

> **Check.** You should see `rolling_mean_lag1_window_size7` first with 52.4, then `rolling_mean_lag1_window_size28` with 22.5 and `lag1` with 6.7.

`rolling_mean_lag1_window_size7` is the mean of the 7 days before the day: the recent level of the product. The promotion flag is not in the top 10. That does not mean that promotions do not matter: a promotion day is rare, so splits on it reduce the total error less. Importance tells you what the model used, not what causes sales.

## 2. LightGBM in the backtest

The next cell wraps the model as a forecaster for `backtest.py` and runs it at the 13 cut-offs. mlforecast forecasts **recursively** by default: it predicts day 1, then uses that forecast as the lag 1 of day 2, and so on.

> **Predict.** Will LightGBM beat ETS (WAPE 41.3%) and the per-product linear regression (41.7%)? Then run the cell (about 20 seconds).

In [ ]:
def lightgbm(lags, lag_transforms, direct=False):
    def forecaster(train, h, future):
        f = make_model(lags, lag_transforms)
        f.fit(train[["unique_id", "ds", "y"] + EXOG], static_features=[], max_horizon=h if direct else None)
        p = f.predict(h, X_df=future[["unique_id", "ds"] + EXOG])
        return p.rename(columns={"lgbm": "yhat"}).assign(yhat=lambda d: d["yhat"].clip(lower=0))
    return forecaster

cost = bt.stock_cost(PRICES)
fc_rec = bt.backtest(sales, lightgbm(**RECURSIVE), h=14, origins=ORIGINS)
rows = [bt.score(fc_rec, sales, "LightGBM, recursive", season=7, cost=cost)]
table = bt.show(rows)

> **Check.** You should see MASE 0.816, WAPE % 40.7, bias % 0.7 and a cost of 14,749 dollars.

## 3. Keep the gap: lags of 14 days and more, or one model per day ahead

A recursive forecast feeds its own forecasts back as inputs. Two other ways use only what is known at the cut-off:

- **Lags of 14 days and more:** every input is at least 14 days old, so it is known for all 14 days of the order.
- **Direct:** `max_horizon=14` trains one model per day ahead. The model for day 5 learns from rows where the inputs are 5 days older than the target.

The direct forecast trains 14 models at each cut-off, so the next cell runs it at the **last 4 cut-offs only** and compares it there with the recursive forecast of section 2.

> **Predict.** Which of the three has the lowest cost at these 4 cut-offs? Then run the cell (about a minute).

In [ ]:
fc_14 = bt.backtest(sales, lightgbm([14, 21, 28], {14: [RollingMean(7), RollingMean(28)]}), h=14, origins=ORIGINS)
print(bt.score(fc_14, sales, "LightGBM, lags of 14 days and more", season=7, cost=cost))

last4 = ORIGINS[-4:]
fc_dir = bt.backtest(sales, lightgbm([1, 7, 14, 28], {1: [RollingMean(7), RollingMean(28)]}, direct=True), h=14, origins=last4)
rows4 = [bt.score(fc_rec[fc_rec["origin"].isin(last4)], sales, "recursive, last 4 cut-offs", season=7, cost=cost),
         bt.score(fc_14[fc_14["origin"].isin(last4)], sales, "lags >= 14, last 4 cut-offs", season=7, cost=cost),
         bt.score(fc_dir, sales, "direct, last 4 cut-offs", season=7, cost=cost)]
table4 = bt.show(rows4)
print("seconds per cut-off: recursive", round(fc_rec.attrs["fit_seconds"] / len(ORIGINS), 1),
      "| direct", round(fc_dir.attrs["fit_seconds"] / len(last4), 1))

> **Check.** You should see the lags of 14 days and more with MASE 0.828 and WAPE % 40.9 (all 13 cut-offs). At the last 4 cut-offs, the costs are: lags of 14 days and more 9,174, recursive 9,756, direct 9,926 dollars. (The `fit s` column of the first two rows is the time of all 13 cut-offs.) The last line shows that the direct forecast takes many times longer per cut-off; the seconds depend on your computer.

Here the direct forecast is the most expensive of the three. Over all 13 cut-offs, the course's backtest table has it at the lowest cost of every method (13,711 dollars), at about 10 times the fit time of the recursive one. Four cut-offs are a small test: the order of the three changes from one cut-off to the next, so compare methods on all 13.

## 4. Failure case: a rolling mean that includes the day

Tomás computes a 7-day rolling mean on the whole series **before** the backtest, with the day itself inside the window. The code looks normal. The next cell trains LightGBM with it, the weekday and the promotion flag.

> **Predict.** Will the leaky model look better or worse than the honest recursive one? Then run the cell.

In [ ]:
leak = sales.sort_values(["unique_id", "ds"]).copy()
leak["rolling7_with_today"] = leak.groupby("unique_id")["y"].transform(lambda s: s.rolling(7, min_periods=1).mean())

def leaky(train, h, future):
    tr = leak[leak["ds"] <= train["ds"].max()]
    X = pd.get_dummies(tr["ds"].dt.dayofweek, prefix="dow").astype(float)
    X["roll"], X["on_promotion"] = tr["rolling7_with_today"].to_numpy(), tr["on_promotion"].to_numpy()
    model = lgb.LGBMRegressor(**LGB).fit(X, tr["y"])
    fu = leak.merge(future[["unique_id", "ds"]], on=["unique_id", "ds"])
    Xf = pd.get_dummies(fu["ds"].dt.dayofweek, prefix="dow").astype(float).reindex(columns=[c for c in X if c.startswith("dow")], fill_value=0.0)
    Xf["roll"], Xf["on_promotion"] = fu["rolling7_with_today"].to_numpy(), fu["on_promotion"].to_numpy()
    return pd.DataFrame({"unique_id": fu["unique_id"], "ds": fu["ds"], "yhat": model.predict(Xf)})

fc_leak = bt.backtest(sales, leaky, h=14, origins=ORIGINS)
print(bt.score(fc_leak, sales, "LightGBM with a leaky rolling mean", season=7, cost=cost))

> **Check.** You should see WAPE % 33.5 and a cost of 4,899 dollars: a third of the honest model's cost.

The leaky model looks far better than every other method. That is the warning sign. For each of the 14 forecast days, its feature contains that day's real sales, which nobody knows on the Sunday before. mlforecast computes its rolling means from lags (`lag_transforms={1: [RollingMean(7)]}` starts from lag 1), so they stop the day before; during a recursive forecast, they use the forecasts. Your own pandas code must do the same.

## 5. Your turn: LightGBM on promotion days

ETS missed 43.5% of the units on the 154 promotion days of the backtest: it cannot see the promotion calendar.

> **Your turn.** In `fc_rec`, keep the rows where the product was on promotion (join `sales[["unique_id", "ds", "on_promotion"]]` on `unique_id` and `ds`). Compute the bias in % of those rows: `100 * (forecast sum - real sum) / real sum`, rounded to 1 decimal. Put it in `promo_bias_lgbm` as a Python float. Then run the check cell.

In [ ]:
p = fc_rec.merge(sales[["unique_id", "ds", "on_promotion"]], on=["unique_id", "ds"])
p = p[p["on_promotion"] == 1]
promo_bias_lgbm = round(float(100 * (p["yhat"].sum() - p["y"].sum()) / p["y"].sum()), 1)
print(promo_bias_lgbm, "% on", len(p), "promotion days")

In [ ]:
expect('promo_bias_lgbm', promo_bias_lgbm, 5.2)

LightGBM is a few percent too high on the promotion days, where ETS was far too low. Its trees can split on the promotion flag after they split on the product's recent level, so the uplift grows with the product. A linear model adds the same number of units to every product.

## Next

You added three LightGBM rows to the backtest table and saw a leak that makes a model look too good. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Deep and foundation models](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.